In [ ]:
import sqlite3
import pandas as pd
import sqlalchemy
from sqlalchemy import create_engine
from pathlib import Path

# Define paths relative to project root
PROJECT_ROOT = Path.cwd()  # Assumes notebook is run from project root
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

# Create directories if they don't exist
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# Connect to the database (update db_filename as needed)
db_filename = "Travis_protaxExport_20250720.db"
db_path = DATA_PROCESSED / db_filename

if not db_path.exists():
    print(f"ERROR: Database not found: {db_path}")
    print(f"Please run JSON_schema_extract.py first to create the database.")
else:
    conn = sqlite3.connect(db_path)
    df = pd.read_sql_query("SELECT * FROM my_table LIMIT 50;", conn)
    print(f"Loaded {len(df)} rows from {db_filename}")

In [40]:
import pandas as pd
import json

# ============================================================================
# STEP 1: Flatten the 'owners' column first
# ============================================================================
print("Step 1: Flattening owners column...")

# Assuming your dataframe is called 'df' and has an 'owners' column
# df = your_dataframe_here

# Parse the owners column (it's a JSON string)
df['owners_parsed'] = df['owners'].apply(
    lambda x: json.loads(x) if pd.notna(x) and x != '' else []
)

# Explode to create one row per owner
df_owners = df.explode('owners_parsed').reset_index(drop=True)

# Normalize the owners dictionaries into columns
owners_df = pd.json_normalize(df_owners['owners_parsed'])

# Remove duplicate pID column from owners_df if it exists
if 'pID' in owners_df.columns:
    owners_df = owners_df.drop('pID', axis=1)

# Combine with pID from original df
df_flattened = pd.concat([
    df_owners[['pID']],
    owners_df
], axis=1)

print(f"Owners flattened: {len(df_flattened)} rows")
print(f"Columns available: {df_flattened.columns.tolist()}\n")

# ============================================================================
# STEP 2: Now flatten exemptions from the flattened owners
# ============================================================================
print("="*70)
print("Step 2: FLATTENING OWNER EXEMPTIONS")
print("="*70)

def parse_exemptions(x):
    """Parse exemptions handling both string and list formats"""
    if isinstance(x, list):
        return x
    if pd.isna(x) or x == '':
        return []
    if isinstance(x, str):
        if x == '[]':
            return []
        try:
            return json.loads(x.replace("'", '"').replace('None', 'null'))
        except:
            return []
    return []

df_flattened['exemptions_parsed'] = df_flattened['exemptions'].apply(parse_exemptions)

# Count total exemptions
total_exemptions = df_flattened['exemptions_parsed'].apply(len).sum()
print(f"Total owner exemptions found: {total_exemptions}")

if total_exemptions > 0:
    # Explode exemptions
    df_exemptions = df_flattened[['pID', 'pAccountID', 'ownerID', 'name', 'exemptions_parsed']].copy()
    df_exemptions = df_exemptions.explode('exemptions_parsed').reset_index(drop=True)
    
    # Filter out empty rows
    df_exemptions = df_exemptions[df_exemptions['exemptions_parsed'].apply(
        lambda x: isinstance(x, dict)
    )]
    
    if len(df_exemptions) > 0:
        # Normalize exemptions
        exemptions_normalized = pd.json_normalize(df_exemptions['exemptions_parsed'])
        
        # Remove any duplicate columns that might exist
        exemptions_cols = [col for col in exemptions_normalized.columns 
                          if col not in ['pID', 'pAccountID', 'ownerID', 'name']]
        
        # Combine
        df_exemptions_final = pd.concat([
            df_exemptions[['pID', 'pAccountID', 'ownerID', 'name']].reset_index(drop=True),
            exemptions_normalized[exemptions_cols].reset_index(drop=True)
        ], axis=1)
        
        print(f"✓ Created dataframe with {len(df_exemptions_final)} owner exemptions")
        print(f"  Columns: {exemptions_cols}")
    else:
        df_exemptions_final = None
        print("  No owner exemptions data to flatten")
else:
    df_exemptions_final = None
    print("  No owner exemptions in this dataset")

# ============================================================================
# STEP 3: Flatten owner value data (homestead breakdowns)
# ============================================================================
print("\n" + "="*70)
print("Step 3: FLATTENING OWNER VALUE DATA (Homestead Breakdowns)")
print("="*70)

df_flattened['ownerValue_parsed'] = df_flattened['ownerValue'].apply(parse_exemptions)

# Count total value records
total_values = df_flattened['ownerValue_parsed'].apply(len).sum()
print(f"Total owner value records found: {total_values}")

if total_values > 0:
    # Explode owner values
    df_values = df_flattened[['pID', 'pAccountID', 'ownerID', 'name', 'ownerValue_parsed']].copy()
    df_values = df_values.explode('ownerValue_parsed').reset_index(drop=True)
    
    # Filter out empty rows
    df_values = df_values[df_values['ownerValue_parsed'].apply(
        lambda x: isinstance(x, dict)
    )]
    
    if len(df_values) > 0:
        # Normalize values
        values_normalized = pd.json_normalize(df_values['ownerValue_parsed'])
        
        # Remove any duplicate columns
        value_cols = [col for col in values_normalized.columns 
                     if col not in ['pID', 'pAccountID', 'ownerID', 'name']]
        
        # Combine
        df_values_final = pd.concat([
            df_values[['pID', 'pAccountID', 'ownerID', 'name']].reset_index(drop=True),
            values_normalized[value_cols].reset_index(drop=True)
        ], axis=1)
        
        print(f"✓ Created dataframe with {len(df_values_final)} owner value records")
        print(f"  Key homestead columns: ownerLandHSValue, ownerLandNHSValue, ownerImprovementHSValue, ownerImprovementNHSValue")
    else:
        df_values_final = None
        print("  No owner value data to flatten")
else:
    df_values_final = None
    print("  No owner value data in this dataset")

# ============================================================================
# STEP 4: Flatten owner taxable data (per-taxing-unit)
# ============================================================================
print("\n" + "="*70)
print("Step 4: FLATTENING OWNER TAXABLE DATA (Per-Taxing-Unit)")
print("="*70)

df_flattened['ownerTaxable_parsed'] = df_flattened['ownerTaxable'].apply(parse_exemptions)

# Count total taxable records
total_taxable = df_flattened['ownerTaxable_parsed'].apply(len).sum()
print(f"Total taxing unit records found: {total_taxable}")

if total_taxable > 0:
    # Explode owner taxable
    df_taxable = df_flattened[['pID', 'pAccountID', 'ownerID', 'name', 'ownerTaxable_parsed']].copy()
    df_taxable = df_taxable.explode('ownerTaxable_parsed').reset_index(drop=True)
    
    # Filter out empty rows
    df_taxable = df_taxable[df_taxable['ownerTaxable_parsed'].apply(
        lambda x: isinstance(x, dict)
    )]
    
    if len(df_taxable) > 0:
        # Normalize taxable
        taxable_normalized = pd.json_normalize(df_taxable['ownerTaxable_parsed'])
        
        # Remove any duplicate columns
        taxable_cols = [col for col in taxable_normalized.columns 
                       if col not in ['pID', 'pAccountID', 'ownerID', 'name']]
        
        # Combine
        df_taxable_final = pd.concat([
            df_taxable[['pID', 'pAccountID', 'ownerID', 'name']].reset_index(drop=True),
            taxable_normalized[taxable_cols].reset_index(drop=True)
        ], axis=1)
        
        print(f"✓ Created dataframe with {len(df_taxable_final)} taxing unit records")
        print(f"  Columns include: taxingUnitID, improvementHSValue, improvementNHSValue, landHSValue, landNHSValue, exemptions")
        
        # Now flatten the exemptions within each taxing unit
        print("\n  Extracting exemptions from taxing units...")
        df_taxable_final['exemptions_parsed'] = df_taxable_final['exemptions'].apply(parse_exemptions)
        
        total_tu_exemptions = df_taxable_final['exemptions_parsed'].apply(len).sum()
        print(f"  Total exemptions across all taxing units: {total_tu_exemptions}")
        
        if total_tu_exemptions > 0:
            # Explode taxing unit exemptions
            df_tu_exemptions = df_taxable_final[['pID', 'pAccountID', 'ownerID', 'name', 
                                                   'taxingUnitID', 'exemptions_parsed']].copy()
            df_tu_exemptions = df_tu_exemptions.explode('exemptions_parsed').reset_index(drop=True)
            
            # Filter out empty rows
            df_tu_exemptions = df_tu_exemptions[df_tu_exemptions['exemptions_parsed'].apply(
                lambda x: isinstance(x, dict)
            )]
            
            if len(df_tu_exemptions) > 0:
                # Normalize exemptions
                tu_exemptions_normalized = pd.json_normalize(df_tu_exemptions['exemptions_parsed'])
                
                # Remove any duplicate columns
                tu_exempt_cols = [col for col in tu_exemptions_normalized.columns 
                                 if col not in ['pID', 'pAccountID', 'ownerID', 'name', 'taxingUnitID']]
                
                # Combine
                df_tu_exemptions_final = pd.concat([
                    df_tu_exemptions[['pID', 'pAccountID', 'ownerID', 'name', 'taxingUnitID']].reset_index(drop=True),
                    tu_exemptions_normalized[tu_exempt_cols].reset_index(drop=True)
                ], axis=1)
                
                print(f"  ✓ Created dataframe with {len(df_tu_exemptions_final)} taxing unit exemptions")
                print(f"    Exemption columns: {tu_exempt_cols}")
            else:
                df_tu_exemptions_final = None
                print("    No taxing unit exemption details to flatten")
        else:
            df_tu_exemptions_final = None
            print("    No exemptions in taxing units")
    else:
        df_taxable_final = None
        df_tu_exemptions_final = None
        print("  No taxable data to flatten")
else:
    df_taxable_final = None
    df_tu_exemptions_final = None
    print("  No taxable data in this dataset")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "="*70)
print("SUMMARY - DataFrames Created")
print("="*70)

dataframes_created = []

# Rename variables to match the promised names
if df_exemptions_final is not None:
    df_exemptions = df_exemptions_final
    dataframes_created.append(("df_exemptions", df_exemptions, "Owner exemptions"))
    print(f"✓ df_exemptions: {len(df_exemptions)} rows, {len(df_exemptions.columns)} columns")
else:
    df_exemptions = None

if df_values_final is not None:
    df_owner_values = df_values_final
    dataframes_created.append(("df_owner_values", df_owner_values, "Homestead value breakdowns"))
    print(f"✓ df_owner_values: {len(df_owner_values)} rows, {len(df_owner_values.columns)} columns")
else:
    df_owner_values = None

if df_taxable_final is not None:
    df_taxable_units = df_taxable_final
    dataframes_created.append(("df_taxable_units", df_taxable_units, "Per-taxing-unit data"))
    print(f"✓ df_taxable_units: {len(df_taxable_units)} rows, {len(df_taxable_units.columns)} columns")
else:
    df_taxable_units = None

if df_tu_exemptions_final is not None:
    df_taxing_unit_exemptions = df_tu_exemptions_final
    dataframes_created.append(("df_taxing_unit_exemptions", df_taxing_unit_exemptions, "Taxing unit exemptions"))
    print(f"✓ df_taxing_unit_exemptions: {len(df_taxing_unit_exemptions)} rows, {len(df_taxing_unit_exemptions.columns)} columns")
else:
    df_taxing_unit_exemptions = None

if len(dataframes_created) == 0:
    print("\n⚠ No exemption data found in this dataset")
    print("  This example property has no homestead exemptions or other exemptions.")

print("\n" + "="*70)
print("Available DataFrames:")
print("="*70)
for name, df_obj, description in dataframes_created:
    print(f"  {name}: {description}")

# The dataframes are now available as:
# - df_exemptions (if exemptions exist)
# - df_owner_values (homestead breakdowns)
# - df_taxable_units (per-taxing-unit)
# - df_taxing_unit_exemptions (if taxing unit exemptions exist)
print("\n✅ All dataframes are now available in your environment!")

Step 1: Flattening owners column...
Owners flattened: 50 rows
Columns available: ['pID', 'pAccountID', 'ownerID', 'ownerPct', 'applyPctExemptions', 'name', 'nameSecondary', 'firstName', 'lastName', 'spouseFirstName', 'spouseLastName', 'addrDeliveryLine', 'addrUnitDesignator', 'addrCity', 'addrZip', 'addrState', 'addrCountry', 'addrFreeForm', 'addrFreeForm1', 'addrFreeForm2', 'addrFreeForm3', 'addrInternational', 'referenceID', 'regTag', 'source', 'cassValidationDt', 'cassValidationBy', 'cassValidationService', 'plus4Code', 'deliveryPoint', 'deliveryPointCheckDigit', 'latitude', 'longitude', 'carrierRoute', 'autoCass', 'agents', 'exemptions', 'ownerValue', 'ownerTaxable']

Step 2: FLATTENING OWNER EXEMPTIONS
Total owner exemptions found: 39
✓ Created dataframe with 39 owner exemptions
  Columns: ['pExemptionID', 'exemptionCode', 'beginDt', 'expirationDt', 'applicationID', 'exemptionComment', 'applyPctExemptions', 'pctExemption', 'qualifyYr', 'grantedDt', 'calculationRule', 'newExemption

In [41]:
df_exemptions

,pID,pAccountID,ownerID,name,pExemptionID,exemptionCode,beginDt,expirationDt,applicationID,exemptionComment,...,pctExemption,qualifyYr,grantedDt,calculationRule,newExemptionIndicator,newExemptionYear,deferralStartDt,deferralEndDt,useBaseYearForAdditionalAmountCalculation,additionalAmountCalculationBaseYear
0,100026,8119592,1610162,AMALA FOUNDATION,6701978,EX-XV,None,None,NaN,None,...,0.0,NaN,None,Standard,0,NaN,None,None,0,0
1,100027,8745341,1537494,RODRIQUEZ JULIO PASTRANO LIFE ESTATE,7246629,HS,None,None,12648.0,Add Exemption HS KH 6/29/23,...,100.0,2021.0,2023-06-29 12:49:06,Standard,0,NaN,None,None,0,0
2,100037,8745339,1676005,MCCORMACK MATT &,7246627,HS,None,None,NaN,None,...,0.0,2017.0,None,Standard,0,NaN,None,None,0,0
3,100038,8745340,1954786,HENF KIRK & MEGAN,7246628,HS,None,None,NaN,,...,100.0,2023.0,2023-01-11 08:32:31,Standard,0,NaN,None,None,0,2024
4,100039,8745337,1859038,VO TAMMY M & HENRY T PAN,7246625,HS,None,None,9713.0,None,...,0.0,2021.0,None,Standard,0,NaN,None,None,0,0
5,100040,8745338,1779841,FUSTES MANUEL &,7246626,HS,None,None,NaN,None,...,0.0,2006.0,None,Standard,0,NaN,None,None,0,0
6,100041,8745342,100024,GONZALEZ ROBERTO JR,7246630,HS,None,None,NaN,None,...,0.0,1993.0,None,Standard,0,NaN,None,None,0,0
7,100042,8745344,100025,ANDERSON DOUGLAS H &,7246632,HS,None,None,NaN,None,...,0.0,1993.0,None,Standard,0,NaN,None,None,0,0
8,100042,8745344,100025,ANDERSON DOUGLAS H &,7246633,OV65,None,None,NaN,None,...,0.0,2014.0,None,Standard,0,NaN,None,None,0,0
9,100044,8745343,174775,GANNON PATRICK T,7246631,HS,None,None,NaN,None,...,0.0,2006.0,None,Standard,0,NaN,None,None,0,0


In [42]:
import pandas as pd

# ============================================================================
# FIND ALL ADDRESSES WITH HOMESTEAD ('HS') EXEMPTION
# ============================================================================

print("="*70)
print("SEARCHING FOR HOMESTEAD EXEMPTIONS")
print("="*70)

# Check if we have the taxable units dataframe (created by previous script)
if 'df_taxable_units' not in globals():
    print("❌ Error: df_taxable_units not found!")
    print("   Please run the flatten_exemptions_from_dataframe_fixed.py script first.")
else:
    print(f"Searching {len(df_taxable_units)} taxing unit records...\n")
    
    # Parse the exemptions within taxing units
    def parse_exemptions(x):
        """Parse exemptions handling both string and list formats"""
        if isinstance(x, list):
            return x
        if pd.isna(x) or x == '':
            return []
        if isinstance(x, str):
            if x == '[]':
                return []
            try:
                import json
                return json.loads(x.replace("'", '"').replace('None', 'null'))
            except:
                return []
        return []
    
    # Parse exemptions
    df_taxable_units['exemptions_parsed'] = df_taxable_units['exemptions'].apply(parse_exemptions)
    
    # Explode to get one row per exemption
    df_exemptions_exploded = df_taxable_units.explode('exemptions_parsed').reset_index(drop=True)
    
    # Filter out rows with no exemptions
    df_exemptions_exploded = df_exemptions_exploded[
        df_exemptions_exploded['exemptions_parsed'].apply(lambda x: isinstance(x, dict))
    ]
    
    if len(df_exemptions_exploded) > 0:
        # Normalize the exemptions
        exemptions_normalized = pd.json_normalize(df_exemptions_exploded['exemptions_parsed'])
        
        # Combine with the main data
        df_all_exemptions = pd.concat([
            df_exemptions_exploded[['pID', 'pAccountID', 'ownerID', 'name', 'taxingUnitID']].reset_index(drop=True),
            exemptions_normalized.reset_index(drop=True)
        ], axis=1)
        
        print(f"Total exemptions found: {len(df_all_exemptions)}")
        print(f"Exemption columns: {exemptions_normalized.columns.tolist()}\n")
        
        # Filter for HS (Homestead) exemptions
        if 'exemptionCode' in df_all_exemptions.columns:
            df_hs = df_all_exemptions[df_all_exemptions['exemptionCode'] == 'HS'].copy()
            
            print(f"Homestead (HS) exemptions found: {len(df_hs)}")
            
            if len(df_hs) > 0:
                # Get unique properties with HS exemptions
                unique_props = df_hs['pID'].nunique()
                print(f"Unique properties with HS exemptions: {unique_props}\n")
                
                # Now get the addresses for these properties
                # We need to flatten the situses (addresses) from the original dataframe
                print("Fetching addresses for properties with HS exemptions...")
                
                # Get the pIDs with HS exemptions
                hs_pids = df_hs['pID'].unique()
                
                # Filter original dataframe for these pIDs
                if 'df' in globals() and 'situses' in df.columns:
                    df_hs_props = df[df['pID'].isin(hs_pids)].copy()
                    
                    # Parse situses
                    df_hs_props['situses_parsed'] = df_hs_props['situses'].apply(
                        lambda x: json.loads(x) if pd.notna(x) and x != '' else []
                    )
                    
                    # Explode situses
                    df_hs_addresses = df_hs_props.explode('situses_parsed').reset_index(drop=True)
                    
                    # Filter valid addresses
                    df_hs_addresses = df_hs_addresses[
                        df_hs_addresses['situses_parsed'].apply(lambda x: isinstance(x, dict))
                    ]
                    
                    if len(df_hs_addresses) > 0:
                        # Normalize addresses
                        import json
                        addresses_normalized = pd.json_normalize(df_hs_addresses['situses_parsed'])
                        
                        # Remove pID from addresses_normalized if it exists
                        if 'pID' in addresses_normalized.columns:
                            addresses_normalized = addresses_normalized.drop('pID', axis=1)
                        
                        # Combine
                        df_hs_addresses_final = pd.concat([
                            df_hs_addresses[['pID']].reset_index(drop=True),
                            addresses_normalized.reset_index(drop=True)
                        ], axis=1)
                        
                        # Merge with exemption data to get owner names and exemption details
                        exemption_cols_to_keep = ['pID', 'name', 'pAccountID', 'ownerID', 'taxingUnitID']
                        
                        # Add exemption-specific columns if they exist
                        exemption_detail_cols = ['exemptionCode', 'exemptionAmt', 'exemptionPct', 
                                                'exemptionValue', 'exemptionType', 'exemptionYear',
                                                'exemptionDescription', 'exemptionStatus']
                        for col in exemption_detail_cols:
                            if col in df_hs.columns:
                                exemption_cols_to_keep.append(col)
                        
                        df_hs_addresses_final = df_hs_addresses_final.merge(
                            df_hs[exemption_cols_to_keep].drop_duplicates(),
                            on='pID',
                            how='left'
                        )
                        
                        # Reorder columns for better readability - exemption info first, then address
                        priority_cols = ['pID', 'name', 'exemptionCode']
                        
                        # Add other exemption columns if they exist
                        for col in ['taxingUnitID', 'exemptionAmt', 'exemptionPct', 'exemptionValue', 
                                   'exemptionType', 'exemptionYear', 'exemptionDescription', 'exemptionStatus']:
                            if col in df_hs_addresses_final.columns and col not in priority_cols:
                                priority_cols.append(col)
                        
                        # Add address columns
                        address_cols = ['streetNum', 'streetPrefix', 'streetName', 'streetSuffix', 
                                       'city', 'state', 'zip']
                        for col in address_cols:
                            if col in df_hs_addresses_final.columns and col not in priority_cols:
                                priority_cols.append(col)
                        
                        # Add remaining columns
                        other_cols = [c for c in df_hs_addresses_final.columns if c not in priority_cols]
                        df_hs_addresses_final = df_hs_addresses_final[priority_cols + other_cols]
                        
                        print(f"✅ Found {len(df_hs_addresses_final)} addresses with HS exemptions")
                        print(f"\nSample addresses:")
                        
                        # Show exemption code and address info
                        display_cols = ['name', 'exemptionCode', 'streetNum', 'streetName', 'city', 'zip']
                        display_cols = [c for c in display_cols if c in df_hs_addresses_final.columns]
                        print(df_hs_addresses_final[display_cols].head())
                        
                        # This is the final dataframe you want
                        df_homestead_addresses = df_hs_addresses_final
                        
                        print("\n" + "="*70)
                        print("✅ SUCCESS - DataFrame created: df_homestead_addresses")
                        print("="*70)
                        print(f"Rows: {len(df_homestead_addresses)}")
                        print(f"Columns: {list(df_homestead_addresses.columns)}")
                        
                    else:
                        print("❌ No valid addresses found in situses data")
                else:
                    print("⚠️  Original 'df' dataframe not found or missing 'situses' column")
                    print("    Creating simplified result with owner info only...")
                    
                    # Create a simplified result with just the exemption data
                    df_homestead_addresses = df_hs.copy()
                    
                    print("\n" + "="*70)
                    print("✅ PARTIAL SUCCESS - DataFrame created: df_homestead_addresses")
                    print("="*70)
                    print(f"Rows: {len(df_homestead_addresses)}")
                    print(f"Contains: Owner names and exemption details (addresses need situses data)")
                    
            else:
                print("❌ No homestead (HS) exemptions found in this dataset")
                df_homestead_addresses = pd.DataFrame()
        else:
            print("❌ 'exemptionCode' column not found in exemptions data")
            print(f"   Available columns: {df_all_exemptions.columns.tolist()}")
            df_homestead_addresses = pd.DataFrame()
    else:
        print("❌ No exemptions found in the taxing units data")
        df_homestead_addresses = pd.DataFrame()

print("\n" + "="*70)

SEARCHING FOR HOMESTEAD EXEMPTIONS
Searching 300 taxing unit records...

Total exemptions found: 573
Exemption columns: ['pPropertyAccountTaxingUnitExemptionID', 'exemptionCode', 'calculationType', 'localExemptionAmount', 'exemptionAmount', 'totalExemptionAmount', 'allocationFactor', 'includeExemptionCount', 'pPropertyAccountTaxingUnitID']

Homestead (HS) exemptions found: 450
Unique properties with HS exemptions: 30

Fetching addresses for properties with HS exemptions...
✅ Found 150 addresses with HS exemptions

Sample addresses:
                                   name exemptionCode streetNum streetName  \
0  RODRIQUEZ JULIO PASTRANO LIFE ESTATE            HS      1008          8   
1  RODRIQUEZ JULIO PASTRANO LIFE ESTATE            HS      1008          8   
2  RODRIQUEZ JULIO PASTRANO LIFE ESTATE            HS      1008          8   
3  RODRIQUEZ JULIO PASTRANO LIFE ESTATE            HS      1008          8   
4  RODRIQUEZ JULIO PASTRANO LIFE ESTATE            HS      1008         

In [43]:
df_homestead_addresses

,pID,name,exemptionCode,taxingUnitID,streetNum,streetPrefix,streetName,streetSuffix,city,state,zip,situsAddressID,primarySitus,streetSecondary,country,international,pAccountID,ownerID
0,100027,RODRIQUEZ JULIO PASTRANO LIFE ESTATE,HS,1001,1008,S,8,ST,None,TX,78704,8503211,1,None,None,None,8745341,1537494
1,100027,RODRIQUEZ JULIO PASTRANO LIFE ESTATE,HS,1002,1008,S,8,ST,None,TX,78704,8503211,1,None,None,None,8745341,1537494
2,100027,RODRIQUEZ JULIO PASTRANO LIFE ESTATE,HS,1003,1008,S,8,ST,None,TX,78704,8503211,1,None,None,None,8745341,1537494
3,100027,RODRIQUEZ JULIO PASTRANO LIFE ESTATE,HS,1034,1008,S,8,ST,None,TX,78704,8503211,1,None,None,None,8745341,1537494
4,100027,RODRIQUEZ JULIO PASTRANO LIFE ESTATE,HS,1097,1008,S,8,ST,None,TX,78704,8503211,1,None,None,None,8745341,1537494
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,100072,CHALLA SWATHI,HS,1001,1102,W,MONROE,ST,None,TX,78704,8503237,1,None,None,None,8745367,1970040
146,100072,CHALLA SWATHI,HS,1002,1102,W,MONROE,ST,None,TX,78704,8503237,1,None,None,None,8745367,1970040
147,100072,CHALLA SWATHI,HS,1003,1102,W,MONROE,ST,None,TX,78704,8503237,1,None,None,None,8745367,1970040
148,100072,CHALLA SWATHI,HS,1034,1102,W,MONROE,ST,None,TX,78704,8503237,1,None,None,None,8745367,1970040


In [44]:
import pandas as pd

print("="*70)
print("CONCATENATING ADDRESS FIELDS")
print("="*70)

if 'df_homestead_addresses' not in globals():
    print("❌ Error: df_homestead_addresses not found!")
    print("   Please run find_homestead_addresses.py first")
else:
    print(f"Working with {len(df_homestead_addresses)} addresses...\n")
    
    def create_full_address(row):
        """Concatenate address fields into a single formatted address"""
        address_parts = []
        
        # Street number
        if pd.notna(row.get('streetNum', None)) and row.get('streetNum', '') != '':
            address_parts.append(str(row['streetNum']))
        
        # Street prefix (N, S, E, W, etc.)
        if pd.notna(row.get('streetPrefix', None)) and row.get('streetPrefix', '') != '':
            address_parts.append(str(row['streetPrefix']))
        
        # Street name
        if pd.notna(row.get('streetName', None)) and row.get('streetName', '') != '':
            address_parts.append(str(row['streetName']))
        
        # Street suffix (St, Ave, Rd, etc.)
        if pd.notna(row.get('streetSuffix', None)) and row.get('streetSuffix', '') != '':
            address_parts.append(str(row['streetSuffix']))
        
        # Street secondary (Unit, Apt, etc.)
        if pd.notna(row.get('streetSecondary', None)) and row.get('streetSecondary', '') != '':
            address_parts.append(str(row['streetSecondary']))
        
        # Join street parts
        street_address = ' '.join(address_parts)
        
        # City, State, Zip
        city_state_zip_parts = []
        
        if pd.notna(row.get('city', None)) and row.get('city', '') != '':
            city_state_zip_parts.append(str(row['city']))
        
        if pd.notna(row.get('state', None)) and row.get('state', '') != '':
            city_state_zip_parts.append(str(row['state']))
        
        if pd.notna(row.get('zip', None)) and row.get('zip', '') != '':
            city_state_zip_parts.append(str(row['zip']))
        
        city_state_zip = ', '.join(city_state_zip_parts)
        
        # Combine all parts
        if street_address and city_state_zip:
            return f"{street_address}, {city_state_zip}"
        elif street_address:
            return street_address
        elif city_state_zip:
            return city_state_zip
        else:
            return ''
    
    # Create the full address column
    df_homestead_addresses['fullAddress'] = df_homestead_addresses.apply(create_full_address, axis=1)
    
    print("✅ Created 'fullAddress' column\n")
    print("Sample addresses:")
    print(df_homestead_addresses[['pID', 'name', 'exemptionCode', 'fullAddress']].head(10))
    
    # Count how many have addresses
    has_address = df_homestead_addresses['fullAddress'].str.len() > 0
    print(f"\n✅ {has_address.sum()} out of {len(df_homestead_addresses)} rows have addresses")
    
    if has_address.sum() < len(df_homestead_addresses):
        print(f"⚠️  {len(df_homestead_addresses) - has_address.sum()} rows are missing address data")
    
    # Optionally create a cleaner version with just key columns
    df_homestead_addresses_simple = df_homestead_addresses[[
        'pID', 'name', 'exemptionCode', 'fullAddress'
    ] + [col for col in ['taxingUnitID', 'exemptionAmt', 'exemptionValue', 'pAccountID', 'ownerID'] 
         if col in df_homestead_addresses.columns]]
    
    print("\n" + "="*70)
    print("SIMPLIFIED DATAFRAME: df_homestead_addresses_simple")
    print("="*70)
    print(f"Columns: {df_homestead_addresses_simple.columns.tolist()}")
    print(f"\nPreview:")
    print(df_homestead_addresses_simple.head())

print("\n✅ Done! Updated df_homestead_addresses with 'fullAddress' column")
print("✅ Created df_homestead_addresses_simple with key columns only")

CONCATENATING ADDRESS FIELDS
Working with 150 addresses...

✅ Created 'fullAddress' column

Sample addresses:
      pID                                  name exemptionCode  \
0  100027  RODRIQUEZ JULIO PASTRANO LIFE ESTATE            HS   
1  100027  RODRIQUEZ JULIO PASTRANO LIFE ESTATE            HS   
2  100027  RODRIQUEZ JULIO PASTRANO LIFE ESTATE            HS   
3  100027  RODRIQUEZ JULIO PASTRANO LIFE ESTATE            HS   
4  100027  RODRIQUEZ JULIO PASTRANO LIFE ESTATE            HS   
5  100037                      MCCORMACK MATT &            HS   
6  100037                      MCCORMACK MATT &            HS   
7  100037                      MCCORMACK MATT &            HS   
8  100037                      MCCORMACK MATT &            HS   
9  100037                      MCCORMACK MATT &            HS   

                   fullAddress  
0       1008 S 8 ST, TX, 78704  
1       1008 S 8 ST, TX, 78704  
2       1008 S 8 ST, TX, 78704  
3       1008 S 8 ST, TX, 78704  
4       1

In [45]:
df_homestead_addresses

,pID,name,exemptionCode,taxingUnitID,streetNum,streetPrefix,streetName,streetSuffix,city,state,zip,situsAddressID,primarySitus,streetSecondary,country,international,pAccountID,ownerID,fullAddress
0,100027,RODRIQUEZ JULIO PASTRANO LIFE ESTATE,HS,1001,1008,S,8,ST,None,TX,78704,8503211,1,None,None,None,8745341,1537494,"1008 S 8 ST, TX, 78704"
1,100027,RODRIQUEZ JULIO PASTRANO LIFE ESTATE,HS,1002,1008,S,8,ST,None,TX,78704,8503211,1,None,None,None,8745341,1537494,"1008 S 8 ST, TX, 78704"
2,100027,RODRIQUEZ JULIO PASTRANO LIFE ESTATE,HS,1003,1008,S,8,ST,None,TX,78704,8503211,1,None,None,None,8745341,1537494,"1008 S 8 ST, TX, 78704"
3,100027,RODRIQUEZ JULIO PASTRANO LIFE ESTATE,HS,1034,1008,S,8,ST,None,TX,78704,8503211,1,None,None,None,8745341,1537494,"1008 S 8 ST, TX, 78704"
4,100027,RODRIQUEZ JULIO PASTRANO LIFE ESTATE,HS,1097,1008,S,8,ST,None,TX,78704,8503211,1,None,None,None,8745341,1537494,"1008 S 8 ST, TX, 78704"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,100072,CHALLA SWATHI,HS,1001,1102,W,MONROE,ST,None,TX,78704,8503237,1,None,None,None,8745367,1970040,"1102 W MONROE ST, TX, 78704"
146,100072,CHALLA SWATHI,HS,1002,1102,W,MONROE,ST,None,TX,78704,8503237,1,None,None,None,8745367,1970040,"1102 W MONROE ST, TX, 78704"
147,100072,CHALLA SWATHI,HS,1003,1102,W,MONROE,ST,None,TX,78704,8503237,1,None,None,None,8745367,1970040,"1102 W MONROE ST, TX, 78704"
148,100072,CHALLA SWATHI,HS,1034,1102,W,MONROE,ST,None,TX,78704,8503237,1,None,None,None,8745367,1970040,"1102 W MONROE ST, TX, 78704"


In [52]:
df_homestead_addresses_distinct = df_homestead_addresses.groupby('fullAddress').agg(list)

In [53]:
df_homestead_addresses_distinct

,pID,name,exemptionCode,taxingUnitID,streetNum,streetPrefix,streetName,streetSuffix,city,state,zip,situsAddressID,primarySitus,streetSecondary,country,international,pAccountID,ownerID
fullAddress,,,,,,,,,,,,,,,,,,
"1000 W MILTON ST, AUSTIN, TX, 78704","[100044, 100044, 100044, 100044, 100044]","[GANNON PATRICK T, GANNON PATRICK T, GANNON PA...","[HS, HS, HS, HS, HS]","[1001, 1002, 1003, 1034, 1097]","[1000, 1000, 1000, 1000, 1000]","[W, W, W, W, W]","[MILTON, MILTON, MILTON, MILTON, MILTON]","[ST, ST, ST, ST, ST]","[AUSTIN, AUSTIN, AUSTIN, AUSTIN, AUSTIN]","[TX, TX, TX, TX, TX]","[78704, 78704, 78704, 78704, 78704]","[8503213, 8503213, 8503213, 8503213, 8503213]","[1, 1, 1, 1, 1]","[None, None, None, None, None]","[None, None, None, None, None]","[None, None, None, None, None]","[8745343, 8745343, 8745343, 8745343, 8745343]","[174775, 174775, 174775, 174775, 174775]"
"1000 W MONROE ST, TX, 78704","[100065, 100065, 100065, 100065, 100065]","[POLLARD ADELA, POLLARD ADELA, POLLARD ADELA, ...","[HS, HS, HS, HS, HS]","[1001, 1002, 1003, 1034, 1097]","[1000, 1000, 1000, 1000, 1000]","[W, W, W, W, W]","[MONROE, MONROE, MONROE, MONROE, MONROE]","[ST, ST, ST, ST, ST]","[None, None, None, None, None]","[TX, TX, TX, TX, TX]","[78704, 78704, 78704, 78704, 78704]","[8503231, 8503231, 8503231, 8503231, 8503231]","[1, 1, 1, 1, 1]","[None, None, None, None, None]","[None, None, None, None, None]","[None, None, None, None, None]","[8745361, 8745361, 8745361, 8745361, 8745361]","[100047, 100047, 100047, 100047, 100047]"
"1002 W MONROE ST, AUSTIN, TX, 78704","[100066, 100066, 100066, 100066, 100066]","[SAAVEDRA MANUEL, SAAVEDRA MANUEL, SAAVEDRA MA...","[HS, HS, HS, HS, HS]","[1001, 1002, 1003, 1034, 1097]","[1002, 1002, 1002, 1002, 1002]","[W, W, W, W, W]","[MONROE, MONROE, MONROE, MONROE, MONROE]","[ST, ST, ST, ST, ST]","[AUSTIN, AUSTIN, AUSTIN, AUSTIN, AUSTIN]","[TX, TX, TX, TX, TX]","[78704, 78704, 78704, 78704, 78704]","[8503232, 8503232, 8503232, 8503232, 8503232]","[1, 1, 1, 1, 1]","[None, None, None, None, None]","[None, None, None, None, None]","[None, None, None, None, None]","[8745362, 8745362, 8745362, 8745362, 8745362]","[100048, 100048, 100048, 100048, 100048]"
"1003 JEWELL ST, AUSTIN, TX, 78704","[100063, 100063, 100063, 100063, 100063]","[YEATON ANDREW, YEATON ANDREW, YEATON ANDREW, ...","[HS, HS, HS, HS, HS]","[1001, 1002, 1003, 1034, 1097]","[1003, 1003, 1003, 1003, 1003]","[None, None, None, None, None]","[JEWELL, JEWELL, JEWELL, JEWELL, JEWELL]","[ST, ST, ST, ST, ST]","[AUSTIN, AUSTIN, AUSTIN, AUSTIN, AUSTIN]","[TX, TX, TX, TX, TX]","[78704, 78704, 78704, 78704, 78704]","[8503229, 8503229, 8503229, 8503229, 8503229]","[1, 1, 1, 1, 1]","[None, None, None, None, None]","[None, None, None, None, None]","[None, None, None, None, None]","[8745359, 8745359, 8745359, 8745359, 8745359]","[1685904, 1685904, 1685904, 1685904, 1685904]"
"1004 W MONROE ST, AUSTIN, TX, 78704","[100067, 100067, 100067, 100067, 100067]","[BENNETT MICHELLE A, BENNETT MICHELLE A, BENNE...","[HS, HS, HS, HS, HS]","[1001, 1002, 1003, 1034, 1097]","[1004, 1004, 1004, 1004, 1004]","[W, W, W, W, W]","[MONROE, MONROE, MONROE, MONROE, MONROE]","[ST, ST, ST, ST, ST]","[AUSTIN, AUSTIN, AUSTIN, AUSTIN, AUSTIN]","[TX, TX, TX, TX, TX]","[78704, 78704, 78704, 78704, 78704]","[8503233, 8503233, 8503233, 8503233, 8503233]","[1, 1, 1, 1, 1]","[None, None, None, None, None]","[None, None, None, None, None]","[None, None, None, None, None]","[8745363, 8745363, 8745363, 8745363, 8745363]","[1527610, 1527610, 1527610, 1527610, 1527610]"
"1005 JEWELL ST, AUSTIN, TX, 78704","[100062, 100062, 100062, 100062, 100062]","[PERCIVAL ALBERT E III & KEVIN CHUCK HUGHES, P...","[HS, HS, HS, HS, HS]","[1001, 1002, 1003, 1034, 1097]","[1005, 1005, 1005, 1005, 1005]","[None, None, None, None, None]","[JEWELL, JEWELL, JEWELL, JEWELL, JEWELL]","[ST, ST, ST, ST, ST]","[AUSTIN, AUSTIN, AUSTIN, AUSTIN, AUSTIN]","[TX, TX, TX, TX, TX]","[78704, 78704, 78704, 78704, 78704]","[8503230, 8503230, 8503230,

In [ ]:
import pandas as pd
import json
from pathlib import Path

print("="*70)
print("CREATING TAXING UNIT DICTIONARY")
print("="*70)

# Define export path relative to project root
PROJECT_ROOT = Path.cwd()  # Assumes notebook is run from project root
DATA_EXPORTS = PROJECT_ROOT / "data" / "exports"
DATA_EXPORTS.mkdir(parents=True, exist_ok=True)

# Method 1: From the flattened taxingunits table
if 'flattened_taxingunits' in [f.replace('.csv', '') for f in globals() if isinstance(globals()[f], str)]:
    print("\nMethod 1: Loading from flattened_taxingunits.csv...")
    try:
        df_taxing_units = pd.read_csv('/mnt/user-data/outputs/flattened_taxingunits.csv')
        
        if 'taxingUnitID' in df_taxing_units.columns and 'taxingUnitName' in df_taxing_units.columns:
            # Create dictionary
            taxing_unit_dict = df_taxing_units[['taxingUnitID', 'taxingUnitName', 'taxingUnitType', 'taxingUnitCode']].drop_duplicates()
            taxing_unit_dict = taxing_unit_dict.sort_values('taxingUnitID')
            
            print(f"✅ Found {len(taxing_unit_dict)} unique taxing units")
            print("\nTaxing Unit Dictionary:")
            print(taxing_unit_dict.to_string(index=False))
            
            # Create a simple lookup dictionary
            tu_lookup = dict(zip(taxing_unit_dict['taxingUnitID'], taxing_unit_dict['taxingUnitName']))
            print(f"\n✅ Created tu_lookup dictionary")
            
        else:
            print("❌ Columns not found in flattened_taxingunits.csv")
    except:
        print("❌ Could not load flattened_taxingunits.csv")

# Method 2: From the original dataframe
if 'df' in globals() and 'taxingunits' in df.columns:
    print("\n" + "="*70)
    print("Method 2: Extracting from original df['taxingunits']...")
    print("="*70)
    
    # Parse taxingunits
    def parse_taxingunits(x):
        if isinstance(x, list):
            return x
        if pd.isna(x) or x == '':
            return []
        if isinstance(x, str):
            if x == '[]':
                return []
            try:
                return json.loads(x.replace("'", '"').replace('None', 'null'))
            except:
                return []
        return []
    
    df['taxingunits_parsed'] = df['taxingunits'].apply(parse_taxingunits)
    
    # Explode
    df_tu = df[['pID', 'taxingunits_parsed']].explode('taxingunits_parsed').reset_index(drop=True)
    
    # Filter valid
    df_tu = df_tu[df_tu['taxingunits_parsed'].apply(lambda x: isinstance(x, dict))]
    
    if len(df_tu) > 0:
        # Normalize
        tu_normalized = pd.json_normalize(df_tu['taxingunits_parsed'])
        
        if 'taxingUnitID' in tu_normalized.columns:
            # Get unique taxing units
            tu_cols = ['taxingUnitID']
            for col in ['taxingUnitName', 'taxingUnitType', 'taxingUnitCode', 'taxingUnitNum']:
                if col in tu_normalized.columns:
                    tu_cols.append(col)
            
            taxing_unit_dict = tu_normalized[tu_cols].drop_duplicates().sort_values('taxingUnitID')
            
            print(f"✅ Found {len(taxing_unit_dict)} unique taxing units from original data")
            print("\nTaxing Unit Dictionary:")
            print(taxing_unit_dict.to_string(index=False))
            
            # Create lookup dictionary
            if 'taxingUnitName' in taxing_unit_dict.columns:
                tu_lookup = dict(zip(taxing_unit_dict['taxingUnitID'], taxing_unit_dict['taxingUnitName']))
                print(f"\n✅ Created tu_lookup dictionary with {len(tu_lookup)} entries")
                print("\nSample lookups:")
                for tu_id in list(tu_lookup.keys())[:5]:
                    print(f"  {tu_id}: {tu_lookup[tu_id]}")
        else:
            print("❌ taxingUnitID not found in normalized data")
    else:
        print("❌ No valid taxing unit data found")

# Method 3: From df_taxable_units (if available)
if 'df_taxable_units' in globals():
    print("\n" + "="*70)
    print("Method 3: From df_taxable_units...")
    print("="*70)
    
    if 'taxingUnitID' in df_taxable_units.columns:
        # Check for name columns in the nested data
        print(f"Columns in df_taxable_units: {df_taxable_units.columns.tolist()}")
        
        # Get unique taxing unit IDs
        unique_tu = df_taxable_units['taxingUnitID'].unique()
        print(f"\nUnique taxing unit IDs found: {sorted(unique_tu)}")

# Method 4: Manual lookup for Travis County (common in Texas)
print("\n" + "="*70)
print("COMMON TRAVIS COUNTY TAXING UNITS (Reference)")
print("="*70)
print("""
Based on your data location (Travis County, TX), here are typical taxing units:

1000 - Travis County
1001 - Travis County General
1002 - School District (varies by location)
1003 - City (Austin, Pflugerville, etc.)
1034 - Travis County Healthcare District / Central Health
1097 - Austin Community College District

Note: The exact names depend on your data source and property location.
""")

# Save the dictionary if we created it
if 'taxing_unit_dict' in locals():
    export_path = DATA_EXPORTS / 'taxing_unit_dictionary.csv'
    taxing_unit_dict.to_csv(export_path, index=False)
    print(f"\n✅ Saved {export_path}")
    
    # Also save as a simple lookup
    if 'tu_lookup' in locals():
        tu_lookup_df = pd.DataFrame(list(tu_lookup.items()), columns=['taxingUnitID', 'taxingUnitName'])
        lookup_path = DATA_EXPORTS / 'taxing_unit_lookup.csv'
        tu_lookup_df.to_csv(lookup_path, index=False)
        print(f"✅ Saved {lookup_path}")

print("\n" + "="*70)
print("USING THE LOOKUP")
print("="*70)
print("""
To use the tu_lookup dictionary:

# Look up a single taxing unit
tu_lookup[1001]  # Returns the name

# Add taxing unit names to your homestead dataframe
df_homestead_addresses['taxingUnitID'].map(tu_lookup)
""")